In [9]:
library("R.matlab")
library("tidyverse")
library("afex")
library("BayesFactor")

In [10]:
extract_metrics <- function(filepath, group) {
  mat <- readMat(filepath)
  if (is.null(mat$meanRT) || is.null(mat$meanMT) ||
      length(mat$meanRT) < 3 || length(mat$meanMT) < 3) {
    message(paste("Skipping file due to missing or invalid data:", filepath))
    return(NULL)
  }
  data.frame(
    Subject = basename(filepath),
    Group = group,
    Block = c('baseline', 'early_learning', 'late_learning'),
    ResponseTime = as.numeric(mat$meanRT[1:3]),
    MovementTime = as.numeric(mat$meanMT[1:3])
  )
}

adult_files <- list.files('adult_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)
child_files <- list.files('children_data', pattern = '\\_Final_Results.mat$', full.names = TRUE)

data_adult <- map_dfr(adult_files, ~extract_metrics(.x, 'adult'))
data_child <- map_dfr(child_files, ~extract_metrics(.x, 'child'))

data_all <- bind_rows(data_adult, data_child)

glimpse(data_all)
head(data_all)

Rows: 72
Columns: 5
$ Subject      <chr> "VML_MEG_011_Final_Results.mat", "VML_MEG_011_Final_Resul…
$ Group        <chr> "adult", "adult", "adult", "adult", "adult", "adult", "ad…
$ Block        <chr> "baseline", "early_learning", "late_learning", "baseline"…
$ ResponseTime <dbl> 0.3590000, 0.3308000, 0.3260000, 0.3590000, 0.3308000, 0.…
$ MovementTime <dbl> 1.0180000, 1.1364667, 1.0935333, 1.0180000, 1.1364667, 1.…


,Subject,Group,Block,ResponseTime,MovementTime
,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,VML_MEG_011_Final_Results.mat,adult,baseline,0.3590,1.018000
2,VML_MEG_011_Final_Results.mat,adult,early_learning,0.3308,1.136467
3,VML_MEG_011_Final_Results.mat,adult,late_learning,0.3260,1.093533
4,VML_MEG_012_2_Final_Results.mat,adult,baseline,0.3590,1.018000
5,VML_MEG_012_2_Final_Results.mat,adult,early_learning,0.3308,1.136467
6,VML_MEG_012_2_Final_Results.mat,adult,late_learning,0.3260,1.093533


In [13]:
# response time anova
anova_rt <- aov_ez(
  id = "Subject",
  dv = "ResponseTime",
  data = data_all,
  between = "Group",
  within = "Block"
)

print(anova_rt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group



Anova Table (Type 3 tests)

Response: ResponseTime
       Effect          df  MSE       F   ges p.value
1       Group       1, 22 0.03 8.95 **  .270    .007
2       Block 1.62, 35.58 0.00    0.73  .003    .461
3 Group:Block 1.62, 35.58 0.00    0.03 <.001    .941
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [ ]:
# movement time anova
anova_mt <- aov_ez(
  id = "Subject",
  dv = "MovementTime",
  data = data_all,
  between = "Group",
  within = "Block"
)

print(anova_mt)


Converting to factor: Group

Contrasts set to contr.sum for the following variables: Group



[1] "2-way mixed ANOVA for Movement Time:"
Anova Table (Type 3 tests)

Response: MovementTime
       Effect          df  MSE         F  ges p.value
1       Group       1, 22 0.02      1.05 .033    .316
2       Block 1.33, 29.34 0.01 24.08 *** .239   <.001
3 Group:Block 1.33, 29.34 0.01      2.34 .030    .129
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘+’ 0.1 ‘ ’ 1

Sphericity correction method: GG 


In [14]:
#response time bayes factor
data_all$Subject <- as.factor(data_all$Subject)
data_all$Group <- as.factor(data_all$Group)
data_all$Block <- as.factor(data_all$Block)

bf_rt <- anovaBF(
  ResponseTime ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)

print(bf_rt)

Bayes factor analysis
--------------
[1] Group + Subject                       : 4.852783  ±1.91%
[2] Block + Subject                       : 0.2134649 ±0.68%
[3] Group + Block + Subject               : 1.40197   ±26.55%
[4] Group + Block + Group:Block + Subject : 0.2238858 ±6.68%

Against denominator:
  ResponseTime ~ Subject 
---
Bayes factor type: BFlinearModel, JZS



In [15]:
#movement time bayes factor
bf_mt <- anovaBF(
  MovementTime ~ Group * Block + Subject,
  data = data_all,
  whichRandom = "Subject"
)

print(bf_mt)

Bayes factor analysis
--------------
[1] Group + Subject                       : 0.5111559 ±1.65%
[2] Block + Subject                       : 159786.9  ±0.76%
[3] Group + Block + Subject               : 94778.56  ±0.9%
[4] Group + Block + Group:Block + Subject : 90650.44  ±6.63%

Against denominator:
  MovementTime ~ Subject 
---
Bayes factor type: BFlinearModel, JZS

